# Effect of Activation Functions on Training Dynamics

**Course:** SWE012 – Deep Learning with Python  
**Topic:** Investigating the effects of activation functions (Sigmoid, Tanh, ReLU, Leaky ReLU) on the training process through controlled experiments.  

**Approach:**
- **Depth:** Systematic experiments on activation functions
- **Breadth:** Interaction of all lecture methodologies (ML fundamentals, regularization, optimization, initialization) with activation functions

---

## 1. Libraries and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 2. Dataset: Fashion-MNIST

Fashion-MNIST is a 10-class image classification problem (28×28 grayscale).  
- **i.i.d. assumption:** Train and test sets are independently sampled from the same distribution.  
- **Generalization:** The model's performance on the **test** set, not on the train set, is our true metric.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

BATCH_SIZE = 128  # Typical minibatch size (sufficient SGD noise)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Train: {len(train_dataset)} | Test: {len(test_dataset)}")
print(f"Number of classes: {len(class_names)} | Input size: 28x28 = 784")
print(f"Minibatch size: {BATCH_SIZE} -> ~{len(train_loader)} iterations per epoch")

In [ ]:
# Sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(class_names[label])
    ax.axis('off')
plt.suptitle('Fashion-MNIST Samples', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Model Architecture and Training Infrastructure

**Feedforward Neural Network:** Input → Hidden Layers → Softmax Output  
- **Softmax + CrossEntropyLoss:** Standard pairing for 10-class classification (logits are passed directly, softmax is inside the loss)  
- **MLE connection:** Minimizing cross-entropy loss = performing MLE under a Categorical distribution  
- **Backpropagation:** Gradients are computed via chain rule + memoization

In [ ]:
ACTIVATION_FUNCTIONS = {
    'Sigmoid': nn.Sigmoid(),
    'Tanh': nn.Tanh(),
    'ReLU': nn.ReLU(),
    'LeakyReLU': nn.LeakyReLU(0.01),
}

def build_model(activation_name, hidden_sizes=[256, 128], use_dropout=False, 
                use_batchnorm=False, input_size=784, num_classes=10):
    """Parametric model builder for controlled experiments."""
    layers = []
    in_features = input_size
    
    for h in hidden_sizes:
        layers.append(nn.Linear(in_features, h))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(h))  # BN: normalize -> scale(gamma) + shift(beta)
        layers.append(ACTIVATION_FUNCTIONS[activation_name].__class__() 
                      if activation_name != 'LeakyReLU' 
                      else nn.LeakyReLU(0.01))
        if use_dropout:
            layers.append(nn.Dropout(0.5))  # p=0.5: drop probability (hidden layers)
        in_features = h
    
    layers.append(nn.Linear(in_features, num_classes))  # Output: raw logits (softmax inside loss)
    return nn.Sequential(*layers)


def init_weights(model, method='he'):
    """Weight initialization strategies."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if method == 'he':
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')  # He: Var = 2/n_in
            elif method == 'xavier':
                nn.init.xavier_normal_(m.weight)  # Xavier: Var = 2/(n_in + n_out)
            elif method == 'random':
                nn.init.normal_(m.weight, 0, 0.5)  # High variance -> exploding risk
            nn.init.zeros_(m.bias)
    return model

In [ ]:
def train_model(model, train_loader, test_loader, optimizer, criterion, epochs=15):
    """Training loop: records train loss, test loss, test accuracy."""
    model.to(device)
    history = {'train_loss': [], 'test_loss': [], 'test_acc': []}
    
    for epoch in range(epochs):
        # --- Training ---
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.view(-1, 784).to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            logits = model(X_batch)           # Forward pass
            loss = criterion(logits, y_batch)  # Compute loss
            loss.backward()                    # Backward pass (backpropagation)
            optimizer.step()                   # Update parameters
            running_loss += loss.item()
        
        train_loss = running_loss / len(train_loader)
        
        # --- Evaluation ---
        model.eval()
        test_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.view(-1, 784).to(device)
                y_batch = y_batch.to(device)
                logits = model(X_batch)
                test_loss += criterion(logits, y_batch).item()
                correct += (logits.argmax(1) == y_batch).sum().item()
                total += y_batch.size(0)
        
        test_loss /= len(test_loader)
        test_acc = correct / total
        
        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
    
    return history

In [ ]:
def plot_results(results_dict, title, ylabel='Loss'):
    """Visualize experiment results."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    for name, hist in results_dict.items():
        axes[0].plot(hist['train_loss'], label=name)
        axes[1].plot(hist['test_loss'], label=name)
        axes[2].plot(hist['test_acc'], label=name)
    
    axes[0].set_title(f'{title} — Train Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend()
    
    axes[1].set_title(f'{title} — Test Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend()
    
    axes[2].set_title(f'{title} — Test Accuracy')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Accuracy')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()


def summary_table(results_dict):
    """Results summary table."""
    print(f"{'Configuration':<30} {'Final Train Loss':>17} {'Final Test Loss':>16} {'Best Test Acc':>15}")
    print('-' * 80)
    for name, hist in results_dict.items():
        best_acc = max(hist['test_acc'])
        print(f"{name:<30} {hist['train_loss'][-1]:>17.4f} {hist['test_loss'][-1]:>16.4f} {best_acc:>14.4f}")

---
## 4. EXPERIMENT 1: Activation Function Comparison (Depth)

**Controlled Experiment Design:** All parameters held constant, only the activation function varies.  
- Architecture: 784 → 256 → 128 → 10  
- Optimizer: Adam (lr=0.001) — industry standard (Momentum + RMSProp + bias correction)  
- Initialization: He (Kaiming) — suitable for the ReLU family  
- Regularization: None (to observe the pure effect of activation)  
- Epochs: 15  

**Expectation:** ReLU and Leaky ReLU will converge faster than Sigmoid and Tanh (due to the vanishing gradient problem).

In [ ]:
EPOCHS = 15
LR = 0.001

results_exp1 = {}
for act_name in ACTIVATION_FUNCTIONS:
    print(f"Training: {act_name}...", end=' ')
    torch.manual_seed(42)
    model = build_model(act_name)
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()  # Softmax + NLL = CCE (10 classes)
    
    history = train_model(model, train_loader, test_loader, optimizer, criterion, EPOCHS)
    results_exp1[act_name] = history
    print(f"Test Acc: {max(history['test_acc']):.4f}")

plot_results(results_exp1, 'Exp 1: Activation Function Comparison')
summary_table(results_exp1)

**Analysis:**
- **Sigmoid/Tanh:** Gradient ≈ 0 in saturation regions → vanishing gradient → slow learning  
- **ReLU:** Gradient = 1 for x > 0 → fast convergence, but **dead neuron** risk (x < 0 → gradient = 0)  
- **Leaky ReLU:** Small gradient (0.01) for x < 0 → mitigates the dead neuron problem  

---
## 5. EXPERIMENT 2: Activation × Optimizer Interaction (Breadth: Optimization)

Let's examine how the optimization methods covered in the lectures interact with activation functions.  

| Optimizer | Key Property |
|-----------|-------------|
| SGD | Basic update: w -= lr × ∇J |
| SGD + Momentum | Velocity accumulation: v = βv + ∇J, reduces oscillation |
| RMSProp | Per-parameter lr: fixed version of AdaGrad |
| Adam | Momentum + RMSProp + bias correction |

In [ ]:
optimizers_config = {
    'SGD': lambda params: optim.SGD(params, lr=0.01),
    'SGD+Momentum': lambda params: optim.SGD(params, lr=0.01, momentum=0.9),
    'RMSProp': lambda params: optim.RMSprop(params, lr=0.001),
    'Adam': lambda params: optim.Adam(params, lr=0.001),
}

# Select the best and worst activations to show optimizer effect
test_activations = ['Sigmoid', 'ReLU']

results_exp2 = {}
for act_name in test_activations:
    for opt_name, opt_fn in optimizers_config.items():
        key = f"{act_name} + {opt_name}"
        print(f"Training: {key}...", end=' ')
        torch.manual_seed(42)
        model = build_model(act_name)
        model = init_weights(model, method='he')
        optimizer = opt_fn(model.parameters())
        criterion = nn.CrossEntropyLoss()
        
        history = train_model(model, train_loader, test_loader, optimizer, criterion, EPOCHS)
        results_exp2[key] = history
        print(f"Test Acc: {max(history['test_acc']):.4f}")

plot_results(results_exp2, 'Exp 2: Activation × Optimizer')
summary_table(results_exp2)

**Analysis:**
- **Sigmoid + SGD:** Worst combination — vanishing gradient + fixed learning rate  
- **Sigmoid + Adam:** Adaptive lr partially compensates for Sigmoid's slowness  
- **ReLU + any optimizer:** Strong gradient flow allows all optimizers to perform well  
- **Momentum effect:** Adding momentum to SGD makes a significant difference, especially with weak-gradient functions like Sigmoid

---
## 6. EXPERIMENT 3: Activation × Regularization (Breadth: Regularization)

Interaction of different regularization techniques with activation functions to control overfitting.

| Method | Mechanism | Bayesian Interpretation |
|--------|-----------|------------------------|
| L2 (Weight Decay) | ½α‖w‖² → shrinks weights | Gaussian prior |
| L1 | α‖w‖₁ → sparsity | Laplace prior |
| Dropout (p=0.5) | Random neuron deactivation → 2ⁿ sub-network ensemble | — |
| Batch Normalization | Normalize at each layer → stable distribution | — |
| Label Smoothing (ε=0.1) | Target: 1-ε correct, ε/(k-1) wrong classes | — |

In [ ]:
# CrossEntropy with Label Smoothing
criterion_smooth = nn.CrossEntropyLoss(label_smoothing=0.1)  # epsilon = 0.1
criterion_normal = nn.CrossEntropyLoss()

reg_configs = {
    'No Reg':        {'dropout': False, 'batchnorm': False, 'wd': 0,    'criterion': criterion_normal},
    'L2 (wd=1e-4)':  {'dropout': False, 'batchnorm': False, 'wd': 1e-4, 'criterion': criterion_normal},
    'Dropout(0.5)':  {'dropout': True,  'batchnorm': False, 'wd': 0,    'criterion': criterion_normal},
    'BatchNorm':     {'dropout': False, 'batchnorm': True,  'wd': 0,    'criterion': criterion_normal},
    'LabelSmooth':   {'dropout': False, 'batchnorm': False, 'wd': 0,    'criterion': criterion_smooth},
}

test_activations_reg = ['Sigmoid', 'ReLU']

results_exp3 = {}
for act_name in test_activations_reg:
    for reg_name, cfg in reg_configs.items():
        key = f"{act_name} + {reg_name}"
        print(f"Training: {key}...", end=' ')
        torch.manual_seed(42)
        model = build_model(act_name, use_dropout=cfg['dropout'], use_batchnorm=cfg['batchnorm'])
        model = init_weights(model, method='he')
        optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=cfg['wd'])
        
        history = train_model(model, train_loader, test_loader, optimizer, cfg['criterion'], EPOCHS)
        results_exp3[key] = history
        print(f"Test Acc: {max(history['test_acc']):.4f}")

plot_results(results_exp3, 'Exp 3: Activation × Regularization')
summary_table(results_exp3)

**Analysis:**
- **Batch Normalization + Sigmoid/Tanh:** BN significantly alleviates the saturation issue by reducing internal covariate shift  
- **Dropout:** Increases train loss (makes learning harder) but closes the generalization gap  
- **L2 vs L1:** L2 shrinks weights but doesn't zero them out; L1 drives irrelevant connections to zero (sparsity)  
- **Label Smoothing:** Prevents overconfident predictions → better calibrated probabilities

---
## 7. EXPERIMENT 4: Activation × Initialization Strategy (Breadth: Initialization)

**Why does this matter?**
- All weights identical → **symmetry problem** (all neurons learn the same thing)  
- Weights too small → **vanishing gradients**  
- Weights too large → **exploding gradients**  

| Method | Variance | Best-Suited Activation |
|--------|---------|----------------------|
| Xavier (Glorot) | 2/(n_in + n_out) | Sigmoid, Tanh |
| He (Kaiming) | 2/n_in | ReLU, Leaky ReLU |
| Random (σ=0.5) | Uncontrolled | — (risky) |

In [ ]:
init_methods = ['xavier', 'he', 'random']

results_exp4 = {}
for act_name in ACTIVATION_FUNCTIONS:
    for init_name in init_methods:
        key = f"{act_name} + {init_name}"
        print(f"Training: {key}...", end=' ')
        torch.manual_seed(42)
        model = build_model(act_name)
        model = init_weights(model, method=init_name)
        optimizer = optim.Adam(model.parameters(), lr=LR)
        criterion = nn.CrossEntropyLoss()
        
        history = train_model(model, train_loader, test_loader, optimizer, criterion, EPOCHS)
        results_exp4[key] = history
        print(f"Test Acc: {max(history['test_acc']):.4f}")

plot_results(results_exp4, 'Exp 4: Activation × Initialization')
summary_table(results_exp4)

**Analysis:**
- **Xavier + Sigmoid/Tanh:** Correct pairing — Xavier preserves signal variance  
- **He + ReLU/LeakyReLU:** Correct pairing — He compensates for ReLU halving the variance with a 2× factor  
- **Random:** Uncontrolled variance → exploding/vanishing risk, especially in deep networks  

---
## 8. Gradient Flow Analysis

Let's directly observe the flow of gradients across layers during backpropagation.  
Vanishing gradient = gradient norm ≈ 0 in deeper layers → learning stalls.

In [ ]:
def get_gradient_norms(model, data_loader, criterion):
    """Compute gradient norm for each layer on a single batch."""
    model.train()
    X_batch, y_batch = next(iter(data_loader))
    X_batch = X_batch.view(-1, 784).to(device)
    y_batch = y_batch.to(device)
    
    model.zero_grad()
    loss = criterion(model(X_batch), y_batch)
    loss.backward()
    
    grad_norms = []
    layer_names = []
    for name, param in model.named_parameters():
        if 'weight' in name and param.grad is not None:
            grad_norms.append(param.grad.norm().item())
            layer_names.append(name)
    return layer_names, grad_norms

# Gradient flow comparison with a deeper network
deep_hidden = [512, 256, 128, 64, 32]  # 5 hidden layers

fig, ax = plt.subplots(figsize=(12, 5))
for act_name in ACTIVATION_FUNCTIONS:
    torch.manual_seed(42)
    model = build_model(act_name, hidden_sizes=deep_hidden)
    model = init_weights(model, method='he')
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    names, norms = get_gradient_norms(model, train_loader, criterion)
    ax.plot(range(len(norms)), norms, 'o-', label=act_name, markersize=6)

ax.set_xlabel('Layer (near input → near output)')
ax.set_ylabel('Gradient Norm')
ax.set_title('Gradient Flow: 5-Layer Network (at Initialization)')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print("\nNote: In Sigmoid/Tanh, gradient norm drops rapidly toward the input layers (vanishing gradient).")
print("In ReLU/LeakyReLU, gradient flow remains much more stable across layers.")

## 9. Dead Neuron Analysis

In ReLU, x < 0 → output = 0 and gradient = 0. If a neuron's input stays consistently negative, that neuron **dies** and can never learn again. Leaky ReLU prevents this with a small negative slope.

In [ ]:
def count_dead_neurons(model, data_loader, num_batches=10):
    """Count neurons that consistently output 0 after ReLU/LeakyReLU."""
    model.eval()
    activations = {}  # layer_idx -> cumulative activation
    counts = {}       # layer_idx -> batch count
    
    hooks = []
    layer_idx = 0
    for module in model.modules():
        if isinstance(module, (nn.ReLU, nn.LeakyReLU)):
            idx = layer_idx
            def hook_fn(m, inp, out, idx=idx):
                act = (out > 0).float().mean(dim=0)  # Activation rate per neuron
                if idx not in activations:
                    activations[idx] = act
                    counts[idx] = 1
                else:
                    activations[idx] += act
                    counts[idx] += 1
            hooks.append(module.register_forward_hook(hook_fn))
            layer_idx += 1
    
    with torch.no_grad():
        for i, (X_batch, _) in enumerate(data_loader):
            if i >= num_batches:
                break
            X_batch = X_batch.view(-1, 784).to(device)
            model(X_batch)
    
    for h in hooks:
        h.remove()
    
    results = {}
    for idx in activations:
        avg_act = activations[idx] / counts[idx]
        dead = (avg_act < 0.01).sum().item()  # Less than 1% active = dead
        total = avg_act.numel()
        results[f'Layer {idx}'] = (dead, total, dead/total*100)
    return results

# ReLU vs LeakyReLU dead neuron comparison
for act_name in ['ReLU', 'LeakyReLU']:
    torch.manual_seed(42)
    model = build_model(act_name)
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    # Short training
    train_model(model, train_loader, test_loader, optimizer, criterion, epochs=10)
    
    dead_info = count_dead_neurons(model, train_loader)
    print(f"\n{act_name} — Dead Neuron Analysis (after 10 epochs):")
    for layer, (dead, total, pct) in dead_info.items():
        print(f"  {layer}: {dead}/{total} dead neurons ({pct:.1f}%)")

---
## 10. Bias-Variance Perspective

The **generalization gap** between train loss and test loss reflects the model's variance (overfitting tendency).  
- **High bias** (underfitting): Both train and test loss are high → insufficient model capacity  
- **High variance** (overfitting): Train loss is low, test loss is high → regularization needed  

The choice of activation function directly affects this balance:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, act_name in enumerate(ACTIVATION_FUNCTIONS):
    hist = results_exp1[act_name]
    axes[i].plot(hist['train_loss'], label='Train Loss', linewidth=2)
    axes[i].plot(hist['test_loss'], label='Test Loss', linewidth=2)
    axes[i].fill_between(range(EPOCHS), hist['train_loss'], hist['test_loss'], alpha=0.2, color='red')
    axes[i].set_title(f'{act_name}\nGen. Gap = {hist["test_loss"][-1] - hist["train_loss"][-1]:.3f}')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss')
    axes[i].legend(fontsize=8)

plt.suptitle('Bias-Variance: Generalization Gap (red area = overfitting indicator)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 11. L1 vs L2 Regularization — Detailed Comparison

- **L2 (Ridge):** J(w) = Loss + α/2 · ‖w‖² → shrinks weights but never zeros them out → Gaussian prior  
- **L1 (Lasso):** J(w) = Loss + α · ‖w‖₁ → drives weights to **exactly zero** → Laplace prior → feature selection  

In [ ]:
# L1 regularization (must be added manually in PyTorch)
def train_with_l1(model, train_loader, test_loader, optimizer, criterion, l1_lambda, epochs=15):
    model.to(device)
    history = {'train_loss': [], 'test_loss': [], 'test_acc': []}
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.view(-1, 784).to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            
            # L1 penalty: alpha * sum(|w|)
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm
            
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        train_loss = running_loss / len(train_loader)
        
        model.eval()
        test_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.view(-1, 784).to(device)
                y_batch = y_batch.to(device)
                logits = model(X_batch)
                test_loss += criterion(logits, y_batch).item()
                correct += (logits.argmax(1) == y_batch).sum().item()
                total += y_batch.size(0)
        
        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss / len(test_loader))
        history['test_acc'].append(correct / total)
    
    return history

# L1 vs L2 comparison with ReLU
results_l1l2 = {}

for reg_type, config in [('No Reg', {'wd': 0, 'l1': 0}), 
                          ('L2 (1e-4)', {'wd': 1e-4, 'l1': 0}),
                          ('L1 (1e-5)', {'wd': 0, 'l1': 1e-5})]:
    torch.manual_seed(42)
    model = build_model('ReLU')
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=config['wd'])
    criterion = nn.CrossEntropyLoss()
    
    if config['l1'] > 0:
        history = train_with_l1(model, train_loader, test_loader, optimizer, criterion, config['l1'], EPOCHS)
    else:
        history = train_model(model, train_loader, test_loader, optimizer, criterion, EPOCHS)
    results_l1l2[reg_type] = history

# Weight distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (reg_type, config) in enumerate([('No Reg', {'wd': 0, 'l1': 0}), 
                                         ('L2', {'wd': 1e-4, 'l1': 0}),
                                         ('L1', {'wd': 0, 'l1': 1e-5})]):
    torch.manual_seed(42)
    model = build_model('ReLU')
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=config['wd'])
    criterion = nn.CrossEntropyLoss()
    if config['l1'] > 0:
        train_with_l1(model, train_loader, test_loader, optimizer, criterion, config['l1'], EPOCHS)
    else:
        train_model(model, train_loader, test_loader, optimizer, criterion, EPOCHS)
    
    all_weights = torch.cat([p.data.cpu().flatten() for p in model.parameters()])
    axes[i].hist(all_weights.numpy(), bins=100, density=True, alpha=0.7)
    axes[i].set_title(f'{reg_type}\nNear-zero weight ratio: {(all_weights.abs() < 0.001).float().mean():.1%}')
    axes[i].set_xlabel('Weight value')
    axes[i].set_xlim(-0.5, 0.5)

plt.suptitle('L1 vs L2: Weight Distributions (L1 → sparsity, L2 → shrinkage)', fontsize=13)
plt.tight_layout()
plt.show()

summary_table(results_l1l2)

---
## 12. Overall Results Summary

In [ ]:
print("=" * 90)
print("OVERALL RESULTS SUMMARY")
print("=" * 90)

print("\n[Exp 1] Activation Function Comparison (Depth):")
summary_table(results_exp1)

print("\n[Exp 2] Activation × Optimizer:")
summary_table(results_exp2)

print("\n[Exp 3] Activation × Regularization:")
summary_table(results_exp3)

print("\n[Exp 4] Activation × Initialization:")
summary_table(results_exp4)

## Conclusions

### Depth: Activation Functions
1. **ReLU/Leaky ReLU** achieve faster convergence and higher accuracy than Sigmoid/Tanh in nearly every configuration.
2. **Sigmoid** shows the slowest learning and lowest performance (vanishing gradient).
3. **Leaky ReLU** mitigates the dead neuron problem of ReLU.

### Breadth: Other Methodologies
4. **Optimizer effect:** Adam provides the most stable performance across all activations. SGD struggles significantly with Sigmoid without momentum.
5. **Batch Normalization** yields a dramatic performance boost when used with Sigmoid/Tanh (reduces internal covariate shift).
6. **Initialization:** He init pairs correctly with ReLU; Xavier init pairs correctly with Sigmoid/Tanh.
7. **L1 → sparsity** (feature selection); **L2 → general shrinkage** (when all features are relevant).
8. **Dropout** is effective at closing the generalization gap but slows down training.

### Practical Recommendation
- Default: **ReLU + He init + Adam + BatchNorm** → safe and strong starting point.
- If dead neuron issue occurs: try **Leaky ReLU** or **ELU**.
- If overfitting occurs: **Dropout + L2** combination is effective.

---
## Course Topics Covered (Checklist)

| Week | Topic | Where Used |
|------|-------|-----------|
| 2 | ML Fundamentals: i.i.d., Generalization, Bias-Variance | Dataset section, Exp 1 analysis, Section 10 |
| 2 | MLE ↔ Loss Function connection | Cross-entropy = Categorical MLE (Section 3) |
| 2 | SGD and Minibatch | All training loops (batch_size=128) |
| 2 | Regularization basics (L1, L2) | Experiment 3, Section 11 (Bayesian interpretation included) |
| 3-4 | Feedforward Networks, Depth vs Width | Model architecture (Section 3) |
| 3-4 | Softmax, Cross-Entropy (BCE/CCE) | Loss function choice (CCE, 10 classes) |
| 3-4 | Backpropagation | Training loop, Gradient flow analysis (Section 8) |
| 3-4 | Activation Functions (Sigmoid, Tanh, ReLU, Leaky ReLU) | **MAIN TOPIC** — All experiments |
| 4 | L2 Weight Decay | Experiment 3 |
| 4 | L1 Lasso (sparsity) | Section 11 |
| 4 | Dropout | Experiment 3 |
| 4 | Label Smoothing | Experiment 3 |
| 4 | Batch Normalization | Experiment 3 |
| 5 | Initialization (Xavier, He) | Experiment 4 |
| 5 | Optimizers (SGD, Momentum, RMSProp, Adam) | Experiment 2 |
| 5 | Vanishing/Exploding Gradients | Gradient flow analysis (Section 8) |